In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Si — neutron powder, time-of-flight, Jorgensen–Von Dreele

In [2]:
import easydiffraction as edi
from easydiffraction import ExperimentFactory
from easydiffraction import StructureFactory
from easydiffraction.analysis import verification as verify

## Build the project

In [3]:
project = edi.Project()

## Define the structure

In [4]:
structure = StructureFactory.from_scratch(name='si')

structure.space_group.name_h_m = 'F d -3 m'  # FullProf Space group symbol
structure.space_group.coord_system_code = '2'

structure.cell.length_a = 5.431342  # FullProf a

structure.atom_sites.create(
    id='Si',  # FullProf Atom
    type_symbol='Si',  # FullProf Typ
    fract_x=0.125,  # FullProf X
    fract_y=0.125,  # FullProf Y
    fract_z=0.125,  # FullProf Z
    adp_type='Biso',  # FullProf Biso
    adp_iso=0.52448,  # FullProf Biso
)

project.structures.add(structure)

## Load the FullProf reference

In [5]:
FULLPROF_PROJECT_DIR = 'pd-neut-tof_jvd_si'
FULLPROF_PRF_FILE = 'arg_si.prf'
FULLPROF_BAC_FILE = 'arg_si.bac'
FULLPROF_SUM_FILE = 'arg_si.sum'
FULLPROF_LABEL = verify.fullprof_label(FULLPROF_PROJECT_DIR, FULLPROF_SUM_FILE)
FULLPROF_ZERO = -9.18766  # FullProf Zero
FULLPROF_SCALE = 0.6750847  # FullProf Scale
FULLPROF_TWOTHETA_BANK = 144.845  # FullProf 2ThetaBank
FULLPROF_DTT1 = 7476.91016  # FullProf Dtt1
FULLPROF_DTT2 = -1.54  # FullProf Dtt2
FULLPROF_SIGMA_0 = 3.5544  # FullProf Sigma-0
FULLPROF_SIGMA_1 = 33.0419  # FullProf Sigma-1
FULLPROF_SIGMA_2 = 0.0  # FullProf Sigma-2
FULLPROF_GAMMA_0 = 0.0  # FullProf Gamma-0
FULLPROF_GAMMA_1 = 2.5430  # FullProf Gamma-1
FULLPROF_GAMMA_2 = 0.0  # FullProf Gamma-2
FULLPROF_ALPHA_0 = 0.0  # FullProf alph0
FULLPROF_ALPHA_1 = 0.597100  # FullProf alph1
FULLPROF_BETA_0 = 0.042210  # FullProf beta0
FULLPROF_BETA_1 = 0.009460  # FullProf beta1

x, calc_fullprof = verify.load_fullprof_calc_profile(
    FULLPROF_PROJECT_DIR,
    FULLPROF_PRF_FILE,
    FULLPROF_BAC_FILE,
    FULLPROF_ZERO,
)

## Create the experiment

In [6]:
experiment = ExperimentFactory.from_scratch(
    name='si',
    sample_form='powder',
    beam_mode='time-of-flight',
    radiation_probe='neutron',
    scattering_type='bragg',
)
verify.set_reference_as_measured(experiment, x, calc_fullprof)

experiment.linked_structures.create(structure_id='si', scale=FULLPROF_SCALE)

experiment.instrument.setup_twotheta_bank = FULLPROF_TWOTHETA_BANK
experiment.instrument.calib_d_to_tof_offset = FULLPROF_ZERO
experiment.instrument.calib_d_to_tof_linear = FULLPROF_DTT1
experiment.instrument.calib_d_to_tof_quadratic = FULLPROF_DTT2

experiment.peak.type = 'jorgensen-von-dreele'
experiment.peak.broad_gauss_sigma_0 = FULLPROF_SIGMA_0
experiment.peak.broad_gauss_sigma_1 = FULLPROF_SIGMA_1
experiment.peak.broad_gauss_sigma_2 = FULLPROF_SIGMA_2
experiment.peak.broad_lorentz_gamma_0 = FULLPROF_GAMMA_0
experiment.peak.broad_lorentz_gamma_1 = FULLPROF_GAMMA_1
experiment.peak.broad_lorentz_gamma_2 = FULLPROF_GAMMA_2
experiment.peak.rise_alpha_0 = FULLPROF_ALPHA_0
experiment.peak.rise_alpha_1 = FULLPROF_ALPHA_1
experiment.peak.decay_beta_0 = FULLPROF_BETA_0
experiment.peak.decay_beta_1 = FULLPROF_BETA_1

experiment.excluded_regions.create(id='1', start=0, end=5000)
experiment.excluded_regions.create(id='2', start=10000, end=100000)

project.experiments.add(experiment)

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • broad_lorentz_gamma_0=0.0                                                                                                    
   • broad_lorentz_gamma_1=0.0                                                                                                    
   • broad_lorentz_gamma_2=0.0                                                                                                    


Peak profile type for experiment 'si' changed to


jorgensen-von-dreele


## edi-cryspy VS FullProf

In [7]:
experiment.calculator.type = 'cryspy'

experiment.linked_structures['si'].scale = FULLPROF_SCALE

project.analysis.calculate()
calc_ed_cryspy = experiment.data.intensity_calc

project.display.pattern_comparison(
    'si',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy,
    reference_label=FULLPROF_LABEL,
    candidate_label='edi-cryspy',
)

Calculator for experiment 'si' already set to


cryspy


## Fit edi-cryspy to FullProf

In [8]:
# experiment.linked_structures['si'].scale = 16.558439186694915
# experiment.peak.broad_lorentz_gamma_1 = 9.998261092381231
experiment.linked_structures['si'].scale.free = True
experiment.peak.broad_lorentz_gamma_1.free = True

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_cryspy_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'si',
    reference=calc_fullprof,
    candidate=calc_ed_cryspy_refined,
    reference_label=FULLPROF_LABEL,
    candidate_label='edi-cryspy (refined)',
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'si' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),17.16
4,🔁 Iterations,106
5,📏 Goodness-of-fit (reduced χ²),299.98
6,"📏 R-factor (Rf, %)",7.45
7,"📏 R-factor squared (Rf², %)",5.16
8,"📏 Weighted R-factor (wR, %)",5.16


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,linked_structure,si,scale,,0.6751,14.4757,0.0245,2044.29 % ↑
2,si,peak,,broad_lorentz_gamma_1,μs/Å,2.5430,0.0000,0.0201,100.00 % ↓


In [9]:
experiment.linked_structures['si'].scale

<si.linked_structure.si.scale = 14.475747765991992 ± 0.024536048669334478 (free=True)>

In [10]:
experiment.peak.broad_lorentz_gamma_1

<si.peak.broad_lorentz_gamma_1 = 3.089161890298719e-08 ± 0.02009969030908651 μs/Å (free=True)>

## edi-crysfml VS FullProf

In [11]:
experiment.calculator.type = 'crysfml'

experiment.linked_structures['si'].scale = FULLPROF_SCALE
experiment.peak.broad_lorentz_gamma_1 = FULLPROF_GAMMA_1

project.analysis.calculate()
calc_ed_crysfml = experiment.data.intensity_calc

project.display.pattern_comparison(
    'si',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml,
    reference_label=FULLPROF_LABEL,
    candidate_label='edi-crysfml',
)

Calculator for experiment 'si' changed to


crysfml


## Fit edi-crysfml to FullProf

In [12]:
# experiment.linked_structures['si'].scale = 1275.028259237954
experiment.linked_structures['si'].scale.free = True
experiment.peak.broad_lorentz_gamma_1.free = False

project.analysis.fit()
project.display.fit.results()

project.analysis.calculate()
calc_ed_crysfml_refined = experiment.data.intensity_calc

project.display.pattern_comparison(
    'si',
    reference=calc_fullprof,
    candidate=calc_ed_crysfml_refined,
    reference_label=FULLPROF_LABEL,
    candidate_label='edi-crysfml (refined)',
)

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'si' for 'single' fitting


🚀 Starting fit process with 'lmfit (leastsq)'...


📈 Goodness-of-fit progress:


...

,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,lmfit (leastsq)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),6.31
4,🔁 Iterations,13
5,📏 Goodness-of-fit (reduced χ²),0.35
6,"📏 R-factor (Rf, %)",0.43
7,"📏 R-factor squared (Rf², %)",0.18
8,"📏 Weighted R-factor (wR, %)",0.18


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,si,linked_structure,si,scale,,0.6751,1144.1576,0.0636,169383.57 % ↑


In [13]:
experiment.linked_structures['si'].scale

<si.linked_structure.si.scale = 1144.1576253616297 ± 0.06363422183287049 (free=True)>

In [14]:
experiment.peak.broad_lorentz_gamma_1

<si.peak.broad_lorentz_gamma_1 = 2.543 ± 0.02009969030908651 μs/Å (free=False)>

## Agreement check

In [15]:
verify.assert_patterns_agree(
    [
        (
            'cryspy vs FullProf',
            verify.restrict_to_included(experiment, calc_fullprof),
            calc_ed_cryspy_refined,
        ),
        (
            'crysfml vs FullProf',
            verify.restrict_to_included(experiment, calc_fullprof),
            calc_ed_crysfml_refined,
        ),
    ],
    raise_on_failure=False,
)

,Comparison,Metric,Expected,Actual,OK
1,cryspy vs FullProf,Profile diff (%),< 2.5,5.16,❌
2,,Max deviation (%),< 6,4.71,✅
3,,Area ratio,0.99 to 1.01,0.9473,❌
4,,Shape correlation,> 0.999,0.9987,❌
5,crysfml vs FullProf,Profile diff (%),< 2.5,0.18,✅
6,,Max deviation (%),< 6,0.10,✅
7,,Area ratio,0.99 to 1.01,1.0030,✅
8,,Shape correlation,> 0.999,1.0000,✅


False